# 04 · Challenging cases (Step 6)

How do frame-only, activity-only and fused retrieval hold up under the conditions named in the proposal? Each test query is tagged automatically, so nothing is labelled by hand:

| Condition | How it is measured |
|---|---|
| Brief / long moments | GT length (under 3 s; over 10 s or over 30% of the video) |
| Crowded scenes | people counted by **YOLOv8n** on the 1-fps frames inside the GT |
| Dark scenes | mean brightness of those frames (darkest 10% of the test set) |
| Viewpoint changes | abrupt drops in similarity between consecutive SigLIP frames inside the GT |
| Multi-action / opposite-action queries | from the query text |

Results reuse the per-query scores saved by Step 4, so the systems are exactly the ones in your ablation table.

**Session settings:** Accelerator **GPU T4 x2** (for YOLO) · Internet **On**
**Inputs (Add Input → Datasets):** `charades-sta-vidsearch` (videos + annotations), `vidsearch-features` (SigLIP frames), `vidsearch-fusion` (Step 4 results).

**Workflow:** run with `DEV = True` first (2–3 minutes), then set `DEV = False` and *Save Version → Save & Run All (Commit)* (about 10 minutes).

In [ ]:
# 1. Get the code, run the tests, install YOLO
from pathlib import Path
import sys, glob, subprocess

REPO_URL = "https://github.com/zankhana46/AI-Video-Investigator.git"
!rm -rf /tmp/repo && git clone -q $REPO_URL /tmp/repo
sys.path.insert(0, "/tmp/repo")
for t in sorted(glob.glob("/tmp/repo/tests/test_*.py")):
    r = subprocess.run([sys.executable, t], capture_output=True, text=True, cwd="/tmp/repo")
    print(r.stdout.strip() or f"(no output from {Path(t).name})")
    if r.returncode:
        print(r.stderr[-2000:])
assert Path("/tmp/repo/vidsearch/scene_tags.py").exists(), "Step 6 files are not on GitHub yet: push them first"

!pip install -q ultralytics

In [ ]:
# 2. Find the attached datasets
import json
from pathlib import Path
IN = Path("/kaggle/input")
print("attached:", sorted(p.name for p in IN.iterdir()))

ann = sorted(IN.rglob("test.jsonl"))
assert ann, "Attach charades-sta-vidsearch"
ANN_DIR = ann[0].parent
tars = [p for p in IN.rglob("videos.tar") if p.is_file()]
if tars:
    !tar -xf "{tars[0]}" -C /tmp
    VID_DIR = Path("/tmp/videos")
else:
    VID_DIR = next(IN.rglob("*.mp4")).parent

siglip = [p for p in IN.rglob("*.npz") if not p.name.startswith(("xclip_", "scene_tags"))]
assert siglip, "Attach vidsearch-features"
FRAME_FEAT_DIR = siglip[0].parent

pq = sorted(IN.rglob("per_query_top1_iou_step4.json"))
assert pq, "Attach vidsearch-fusion (Step 4 output)"
PQ_PATH = pq[0]

print("annotations:", ANN_DIR)
print("videos:     ", VID_DIR, "| count:", len(list(VID_DIR.glob("*.mp4"))))
print("SigLIP:     ", FRAME_FEAT_DIR)
print("Step 4:     ", PQ_PATH)

In [ ]:
# 3. Settings
import numpy as np
DEV = True                     # True: 20 test videos (checks that everything works). False: all 1,334.
DEV_VIDEOS = 20
YOLO_WEIGHTS = "yolov8n.pt"    # the small YOLO from the plan
CONF = 0.25                    # YOLO detection confidence
NO_PERSON_MAX = 0.5            # "no person visible": fewer than 0.5 people on average in the GT
MULTI_MIN, CROWD_MIN = 2, 3    # "2+ people" and "crowded (3+ people)": average count in the GT
DARK_SHARE = 0.10              # darkest 10% of test moments
CHANGE_PCT = 99                # a "viewpoint change" = a frame-to-frame change above the 99th percentile
MIN_N = 30                     # categories smaller than this are flagged "too few to judge"
BOOT = 5000

WORK = Path("/kaggle/working")
RES_OUT = WORK / "results"
RES_OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
# 4. Load the test queries and Step 4's per-query results
import pandas as pd
from vidsearch import curves as cv, fusion as fu, scene_tags as st

test = cv.load_jsonl(ANN_DIR / "test.jsonl", "test")
if DEV:
    keep = set(cv.videos_of(test)[:DEV_VIDEOS])
    test = [r for r in test if r["vid"] in keep]
PQ = json.load(open(PQ_PATH))
SYSTEMS = {"frame": "frame_hybrid", "activity": "activity_hybrid", "fused": "fused_hybrid"}
missing = [k for k in SYSTEMS.values() if k not in PQ]
assert not missing, f"Step 4 results lack {missing}"
assert all(r["qid"] in PQ["fused_hybrid"] for r in test), "query ids do not match Step 4"

videos = [(v, VID_DIR / f"{v}.mp4") for v in cv.videos_of(test)]
print(f"{len(test)} test queries on {len(videos)} videos")
for s, k in SYSTEMS.items():
    print(f"  {s:8s} R1@0.5 on these queries: {100 * np.mean([PQ[k][r['qid']] >= 0.5 for r in test]):.2f}")

In [ ]:
# 5. Count people with YOLO and measure brightness on every 1-fps frame (the GPU part)
import torch, time
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else "cpu"
try:
    yolo = YOLO(YOLO_WEIGHTS)
except Exception as e:                       # very old/new ultralytics: fall back to its current small model
    print("could not load", YOLO_WEIGHTS, "->", e, "| using yolo11n.pt")
    YOLO_WEIGHTS = "yolo11n.pt"; yolo = YOLO(YOLO_WEIGHTS)

TAGS, failed, stats = st.tag_videos(yolo, videos, device=DEVICE, conf=CONF)
st.save_tags(RES_OUT / "scene_tags_test.npz", TAGS)
print(stats)
print("failed:", failed[:5])
v0 = next(iter(TAGS))
print(f"example {v0}: people per second {TAGS[v0]['persons'][:15].astype(int).tolist()}")

In [ ]:
# 6. Viewpoint changes from the cached SigLIP frames (CPU)
FT = st.load_frame_table(FRAME_FEAT_DIR, cv.videos_of(test))
assert FT, "SigLIP frame features not found"
CHANGES = {v: st.visual_change(F) for v, F in FT.items()}
all_d = np.concatenate([d for d in CHANGES.values() if len(d)])
CHANGE_THR = float(np.percentile(all_d, CHANGE_PCT))
print(f"frame-to-frame change: median {np.median(all_d):.3f}, {CHANGE_PCT}th percentile {CHANGE_THR:.3f}")

In [ ]:
# 7. Tag every query
Q = pd.DataFrame(st.query_tags(test, TAGS, CHANGES))
dur_cat = np.array([fu.duration_category(r) for r in test])
DARK_THR = float(Q.bright.quantile(DARK_SHARE))
opp = {q for _, a, b in cv.find_opposite_pairs(test) for q in (a["qid"], b["qid"])}

CATS = {
    "all test queries": np.ones(len(Q), bool),
    "brief (<3 s)": dur_cat == "brief (<3 s)",
    "medium": dur_cat == "medium",
    "long (>10 s or >30%)": dur_cat == "long (>10 s or >30%)",
    "no person visible": (Q.persons_mean < NO_PERSON_MAX).values,
    "2+ people": (Q.persons_mean >= MULTI_MIN).values,
    "crowded (3+ people)": (Q.persons_mean >= CROWD_MIN).values,
    f"darkest {int(DARK_SHARE * 100)}%": (Q.bright <= DARK_THR).values,
    "viewpoint change inside GT": (Q.max_change > CHANGE_THR).values,
    "multi-action query": Q.multi_action.values.astype(bool),
    "opposite-action query": Q.qid.isin(opp).values,
}
for name, m in CATS.items():
    Q[name] = m
Q.to_csv(RES_OUT / "query_tags_test.csv", index=False)
print(f"brightness threshold for 'darkest {int(DARK_SHARE * 100)}%': {DARK_THR:.1f} / 255")
print("people in the GT (average count): ", Q.persons_mean.round().value_counts().sort_index().to_dict())
display(pd.Series({k: int(v.sum()) for k, v in CATS.items()}, name="queries"))

In [ ]:
# 8. Accuracy per category, with 95% CIs for fused minus frame (resampling whole videos)
IOU = {s: np.array([PQ[k][q] for q in Q.qid]) for s, k in SYSTEMS.items()}
groups = Q.vid.values
ROWS = []
for name, m in CATS.items():
    n = int(m.sum())
    row = {"category": name, "n": n}
    for s in SYSTEMS:
        row[f"{s} R1@0.5"] = round(100 * float((IOU[s][m] >= 0.5).mean()), 1) if n else np.nan
    if n >= 2:
        b = fu.cluster_bootstrap(IOU["fused"][m] >= 0.5, IOU["frame"][m] >= 0.5, groups[m], B=BOOT)
        row.update({"fused - frame": b["diff"], "95% CI": f"[{b['ci_low']}, {b['ci_high']}]",
                    "significant": bool(b["ci_low"] > 0 or b["ci_high"] < 0)})
    row["note"] = "too few to judge" if n < MIN_N else ""
    ROWS.append(row)
CAT_TABLE = pd.DataFrame(ROWS)
display(CAT_TABLE)
CAT_TABLE.to_csv(RES_OUT / "challenging_cases_test.csv", index=False)

In [ ]:
# 9. Figure for the report: R1@0.5 per category, with 95% CIs per system
import matplotlib.pyplot as plt
names = [n for n, m in CATS.items() if m.sum() >= 2]
colors = {"frame": "tab:blue", "activity": "tab:orange", "fused": "black"}
fig, ax = plt.subplots(figsize=(9, 0.55 * len(names) + 1.5))
y = np.arange(len(names))
for j, s in enumerate(SYSTEMS):
    vals, lo, hi = [], [], []
    for name in names:
        m = CATS[name]
        b = fu.cluster_bootstrap(IOU[s][m] >= 0.5, np.zeros(m.sum()), groups[m], B=1000)
        vals.append(b["diff"]); lo.append(b["diff"] - b["ci_low"]); hi.append(b["ci_high"] - b["diff"])
    ax.barh(y + (j - 1) * 0.27, vals, 0.27, xerr=[lo, hi], color=colors[s], alpha=0.85, label=s,
            error_kw={"lw": 0.8, "capsize": 2})
ax.set_yticks(y); ax.set_yticklabels([f"{n}  (n={int(CATS[n].sum())})" for n in names], fontsize=8)
ax.invert_yaxis(); ax.set_xlabel("R1@0.5 (%) on test"); ax.grid(axis="x", alpha=0.3); ax.legend(loc="lower right")
plt.tight_layout(); plt.savefig(RES_OUT / "challenging_cases_test.png", dpi=150); plt.show()

In [ ]:
# 10. One example frame per condition (people boxes drawn by YOLO)
import textwrap
path_of = dict(videos)
rng = np.random.default_rng(0)
def mid(r): return (r["start"] + r["end"]) / 2
def show(ax, img, title):
    if img is None:
        ax.text(0.5, 0.5, "no example", ha="center"); ax.axis("off"); return
    ax.imshow(img[..., ::-1]); ax.set_title(title, fontsize=8); ax.axis("off")
recs_by_q = {r["qid"]: r for r in test}

panels = []
def add(img, title, r):
    panels.append((img, f"{title}\n'{textwrap.shorten(r['query'], 45)}'"))
cand = Q[CATS["no person visible"]]
if len(cand):
    r = recs_by_q[cand.qid.iloc[rng.integers(len(cand))]]
    add(st.frame_at(path_of[r["vid"]], mid(r)), "no person visible", r)
if Q.persons_mean.notna().any():
    r = recs_by_q[Q.loc[Q.persons_mean.idxmax(), "qid"]]
    f = st.frame_at(path_of[r["vid"]], mid(r))
    img = yolo.predict(f, classes=[0], conf=CONF, verbose=False)[0].plot() if f is not None else None
    add(img, f"most people: {Q.persons_mean.max():.1f} on average", r)
if Q.bright.notna().any():
    r = recs_by_q[Q.loc[Q.bright.idxmin(), "qid"]]
    add(st.frame_at(path_of[r["vid"]], mid(r)), f"darkest moment: brightness {Q.bright.min():.0f}/255", r)
if Q.max_change.notna().any():
    r = recs_by_q[Q.loc[Q.max_change.idxmax(), "qid"]]
    d = CHANGES[r["vid"]]; b = st.gt_bins(r, len(d) + 1)
    i = max((k for k in b if k + 1 in set(b.tolist())), key=lambda k: d[k], default=int(b[0]))
    add(st.frame_at(path_of[r["vid"]], i + 0.5), f"viewpoint change: before ({i + 0.5:.1f} s)", r)
    add(st.frame_at(path_of[r["vid"]], i + 1.5), f"after ({i + 1.5:.1f} s), change {d[i]:.2f}", r)
cand = [recs_by_q[q] for q in Q.qid[CATS["brief (<3 s)"]]]
if cand:
    r = cand[rng.integers(len(cand))]
    add(st.frame_at(path_of[r["vid"]], mid(r)), f"brief moment: {r['end'] - r['start']:.1f} s", r)

fig, axes = plt.subplots(2, 3, figsize=(13, 6.5))
for ax, (img, title) in zip(axes.flat, panels):
    show(ax, img, title)
for ax in list(axes.flat)[len(panels):]:
    ax.axis("off")
plt.tight_layout(); plt.savefig(RES_OUT / "challenging_examples.png", dpi=130); plt.show()

In [ ]:
# 11. Summary file and a report-ready table
lines = ["| Condition | n | Frame | Activity | Fused | Fused − frame [95% CI] |", "|---|---|---|---|---|---|"]
for r in ROWS:
    ci = f"{r.get('fused - frame', '–')} {r.get('95% CI', '')}".strip()
    if r.get("significant"):
        ci = f"**{ci}**"
    flag = " (few)" if r["n"] < MIN_N else ""
    val = lambda k: "–" if r["n"] == 0 else r[k]
    lines.append(f"| {r['category']}{flag} | {r['n']} | {val('frame R1@0.5')} | {val('activity R1@0.5')} | {val('fused R1@0.5')} | {ci} |")
(RES_OUT / "challenging_cases_table.md").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

overall = ROWS[0]["fused R1@0.5"]
hard = [r["category"] for r in ROWS[1:] if r["n"] >= MIN_N and r["fused R1@0.5"] < overall - 5]
helped = [r["category"] for r in ROWS[1:] if r.get("significant") and r["fused - frame"] > 0]
print("\nhardest conditions (fused R1@0.5 more than 5 points below overall):", hard or "none")
print("conditions where fusion significantly beats frame:", helped or "none")
json.dump({"dev": DEV, "yolo": YOLO_WEIGHTS, "conf": CONF, "thresholds": {
               "no_person_max": NO_PERSON_MAX, "multi_min": MULTI_MIN, "crowd_min": CROWD_MIN,
               "dark_brightness": DARK_THR, "change_threshold": CHANGE_THR, "change_pct": CHANGE_PCT},
           "tagging": stats, "categories": ROWS, "hardest": hard, "fusion_helps": helped},
          open(RES_OUT / "step6_summary.json", "w"), indent=2, default=str)
!ls {RES_OUT}

### Full run
1. Set `DEV = False` in cell 3.
2. **Save Version → Save & Run All (Commit)**, then stop the interactive session (power icon).
3. When it finishes, the report material is in the version's **Output → results/**: `challenging_cases_table.md`, `challenging_cases_test.png` and `challenging_examples.png`. No dataset is needed, as no later step reads these files.